In [ ]:
import numpy as np, cv2, torch, tensorflow as tf, sklearn, matplotlib.pyplot as mpl
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers, models
from tensorflow.keras.layers import Dense, Conv2D, MaxPool2D, Flatten, Dropout
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from pathlib import Path

In [ ]:
root = Path('./MEDIAPIPE_DATA/processed_combine_asl_dataset/')
skip = {'j', 'z'}
flip = {'c'}
image_set = []
for subfolder in sorted(root.iterdir()):
    if not subfolder.is_dir():
        continue
    if subfolder.name.lower() in skip:
        continue

    exts = {'.jpg', '.jpeg', '.png'}
    for file in sorted(subfolder.iterdir()):
        if file.suffix.lower() not in exts:
            continue
        label = subfolder.name
        image = cv2.imread(str(file))
        if image is None:
            continue
        image = cv2.resize(image, (128, 128))
        if subfolder.name.lower() in flip:
            image = cv2.flip(image, 1)
        image_set.append((label, image))
labels, images = zip(*image_set)
X = np.array(images)
y = np.array(labels)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

In [ ]:
y_test = np.array([(ord(l.lower()) - ord('a')) for l in y_test])
y_train = np.array([(ord(l.lower()) - ord('a')) for l in y_train])
y_test_cat = to_categorical(y_test, 26)
y_train_cat = to_categorical(y_train, 26)

In [4]:
data_augmentation = models.Sequential([
    layers.RandomZoom(0.25, 0.25),
])
model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),
    layers.Rescaling(1./255),
    data_augmentation,
    
    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPool2D(),                    

    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPool2D(),                    

    layers.Conv2D(128, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPool2D(),                    

    layers.Conv2D(256, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPool2D(),                    

    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(26, activation='softmax'),
])

model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.Adam(5e-4), metrics=['accuracy'])
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=8,
                                     restore_best_weights=True),
]

In [ ]:
for i in range (15):
    model.fit(X_train, y_train_cat, epochs=1)
    predictions = model.predict(X_test)
    predictions = np.argmax(predictions, axis=1) 
    print(classification_report(y_test, predictions))
    model.save(f'./MEDIAPIPE2_Model{i}.keras')

933/933 [==============================] - 151s 162ms/step
              precision    recall  f1-score   support

           0       1.00      0.01      0.03      1147
           1       1.00      0.70      0.82       987
           2       1.00      0.26      0.41       697
           3       0.33      0.96      0.49      1492
           4       1.00      0.13      0.23      1174
           5       0.68      0.96      0.80      1523
           6       0.95      0.95      0.95      1623
           7       0.41      1.00      0.58      1615
           8       0.99      0.52      0.68      1415
          10       0.64      0.42      0.51      1636
          11       1.00      0.29      0.45      1605
          12       1.00      0.02      0.04       571
          13       0.97      0.77      0.86       725
          14       0.18      0.99      0.31      1203
          15       0.89      0.94      0.92      1041
          16       0.93      0.95      0.94      1102
          17       0.6

In [ ]:
model = tf.keras.models.load_model('./MEDIAPIPE2_Model6.keras')